In [1]:
import os
import json
import sqlite3
import gradio as gr
from openai import OpenAI

In [2]:
DB = "hotel.db"
sql_schema = """
CREATE TABLE IF NOT EXISTS rooms (
    id INTEGER PRIMARY KEY,
    size TEXT,
    room_type TEXT,
    is_available BOOLEAN,
    name TEXT UNIQUE,
    price INTEGER,
    modified_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
"""

rooms_data = [
    ("250 sqft", "single", True, "Room 101", 5000),
    ("350 sqft", "double", True, "Room 102", 7500),
    ("450 sqft", "deluxe", False, "Room 103", 10000),
    ("500 sqft", "suite", True, "Room 104", 15000),
    ("300 sqft", "single", False, "Room 105", 5500),
    ("400 sqft", "double", True, "Room 106", 8000),
    ("550 sqft", "deluxe", True, "Room 107", 12000),
    ("600 sqft", "suite", False, "Room 108", 18000),
    ("280 sqft", "single", True, "Room 109", 4500),
    ("420 sqft", "double", False, "Room 110", 8500)
]

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()

    cursor.execute(sql_schema)

    cursor.executemany("""
        INSERT OR IGNORE INTO rooms (size, room_type, is_available, name, price)
        VALUES (?, ?, ?, ?, ?)
    """, rooms_data)

    conn.commit()

print("10 room records inserted successfully!")

10 room records inserted successfully!


In [3]:
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
url = "https://api.anthropic.com/v1/"
openai = OpenAI(api_key=anthropic_api_key, base_url=url)

In [4]:
def get_available_rooms():
    print("Fetching available rooms from the database...")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM rooms WHERE is_available = 1")
        available_rooms = cursor.fetchall()
    return f"Available rooms: {available_rooms}" if available_rooms else "No available rooms at the moment."

In [5]:
def get_room_type():
    print(f"Fetching room types available at the hotel...")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT DISTINCT room_type FROM rooms")
        room_types = cursor.fetchall()
    return f"Available room types: {', '.join([rt[0] for rt in room_types])}" if room_types else "No room types available at the moment."

In [6]:
def check_availability_for_double_room():
    print("Checking availability for double rooms...")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM rooms WHERE room_type = 'Double' AND is_available = 1")
        available_double_rooms = cursor.fetchall()
    return f"Available double rooms: {available_double_rooms}" if available_double_rooms else "No available double rooms at the moment."

In [7]:
room_type_enum = ["single", "double", "deluxe", "suite"]
def check_room_price_range(room_type):
    print(f"Checking price range for {room_type} rooms...")
    if(room_type.lower() not in room_type_enum):
        return f"Invalid room type. Please choose from: {', '.join(room_type_enum)}"
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT MIN(price), MAX(price) FROM rooms WHERE room_type = ?", (room_type.lower(),))
        price_range = cursor.fetchone()
    return f"The price range for {room_type} rooms is: ${price_range[0]} - ${price_range[1]}" if price_range else f"No price information available for {room_type} rooms."

In [8]:
get_available_rooms_function = {
    "name": "get_available_rooms",
    "description": "Fetches a list of available rooms from the hotel database.",
    "parameters": {
        "type": "object",
        "properties": {}
    }
}
get_room_type_function = {
    "name": "get_room_type",
    "description": "Fetches the types of rooms available at the hotel.",
    "parameters": {
        "type": "object",
        "properties": {}
    }
}
check_availability_for_double_room_function = {
    "name": "check_availability_for_double_room",
    "description": "Checks if there are any available double rooms in the hotel.",
    "parameters": {
        "type": "object",
        "properties": {}
    }
}
check_room_price_range_function = {
    "name": "check_room_price_range",
    "description": "Checks the price range for a specific type of room in the hotel.",
    "parameters": {
        "type": "object",
        "properties": {
            "room_type": {
                "type": "string",
                "description": "The type of room to check the price range for (e.g.,'Single', 'Double', 'Deluxe', 'Suite')."
            }
        },
        "required": ["room_type"]
    }
}

In [9]:
tools = [{"type": "function", "function": get_available_rooms_function},
         {"type": "function", "function": get_room_type_function},
         {"type": "function", "function": check_availability_for_double_room_function},
         {"type": "function", "function": check_room_price_range_function}]

In [10]:
system_message = "You are a helpful assistant that provides information about hotel rooms, their availability, and pricing. You can also check the status of flights if needed."
MODEL = "claude-haiku-5-5"

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print(f"Messages: {messages}")
    response = openai.chat.completions.create(model=MODEL, reasoning_effort="none", messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.extend(response)
        response = openai.chat.completions.create(model=MODEL, reasoning_effort="none", messages=messages)
        
    return response.choices[0].message.content

In [11]:
def handle_tool_call(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_available_rooms":
            available_rooms = get_available_rooms()
            responses.append({
                "role": "tool",
                "content": available_rooms,
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "get_room_type":
            room_types = get_room_type()
            responses.append({
                "role": "tool",
                "content": room_types,
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "check_availability_for_double_room":
            available_double_rooms = check_availability_for_double_room()
            responses.append({
                "role": "tool",
                "content": available_double_rooms,
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "check_room_price_range":
            arguments = json.loads(tool_call.function.arguments)
            room_type = arguments.get('room_type')
            price_details = check_room_price_range(room_type)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [12]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Messages: [{'role': 'system', 'content': 'You are a helpful assistant that provides information about hotel rooms, their availability, and pricing. You can also check the status of flights if needed.'}, {'role': 'user', 'content': 'Hey yo'}]
Messages: [{'role': 'system', 'content': 'You are a helpful assistant that provides information about hotel rooms, their availability, and pricing. You can also check the status of flights if needed.'}, {'role': 'user', 'content': 'Hey yo'}, {'role': 'assistant', 'content': 'Hey! I can help with hotel rooms: checking what room types are available, whether double rooms are open, current availability, and price ranges. What are you looking for?'}, {'role': 'user', 'content': 'are there any available rooms?'}]
Fetching available rooms from the database...
Messages: [{'role': 'system', 'content': 'You are a helpful assistant that provides information about hotel rooms, their availability, and pricing. You can also check the status of flights if needed.